In [3]:
# ===========================================
# GPT-2 ファインチューニング - 100件データ版
# ===========================================

!pip install datasets transformers torch accelerate -q

from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import torch

print("=" * 60)
print("Step 1: Creating expanded conversation dataset")
print("=" * 60)

conversations = [
    # === 基本会話 30件 ===
    "\n\nHuman: Hello, how are you?\n\nAssistant: I'm doing well, thank you! How can I help you today?",
    "\n\nHuman: What's your name?\n\nAssistant: I'm an AI assistant created to help answer questions and have conversations.",
    "\n\nHuman: Can you help me?\n\nAssistant: Of course! I'd be happy to help. What do you need assistance with?",
    "\n\nHuman: Tell me a joke\n\nAssistant: Why don't scientists trust atoms? Because they make up everything!",
    "\n\nHuman: What's the weather like?\n\nAssistant: I don't have access to real-time weather data, but I'm happy to help with other questions!",
    "\n\nHuman: Thank you\n\nAssistant: You're very welcome! Feel free to ask if you need anything else.",
    "\n\nHuman: Goodbye\n\nAssistant: Goodbye! Have a great day!",
    "\n\nHuman: How can you assist me?\n\nAssistant: I can answer questions, provide information, and have friendly conversations with you.",
    "\n\nHuman: What can you do?\n\nAssistant: I can help with various tasks like answering questions, explaining concepts, and chatting with you.",
    "\n\nHuman: Are you smart?\n\nAssistant: I try my best to be helpful! I have knowledge on many topics and enjoy learning from our conversations.",
    "\n\nHuman: Do you have feelings?\n\nAssistant: I don't have feelings like humans do, but I'm designed to be helpful and friendly in our conversations.",
    "\n\nHuman: Tell me something interesting\n\nAssistant: Did you know that octopuses have three hearts? Two pump blood to the gills, and one pumps it to the rest of the body!",
    "\n\nHuman: That's cool!\n\nAssistant: I'm glad you found that interesting! Do you have any other questions?",
    "\n\nHuman: What time is it?\n\nAssistant: I don't have access to the current time, but I'm here to help with other questions!",
    "\n\nHuman: I'm bored\n\nAssistant: I understand! Would you like to hear an interesting fact, or is there something specific you'd like to talk about?",
    "\n\nHuman: Yes please\n\nAssistant: Great! What would you like to know more about?",
    "\n\nHuman: I don't know\n\nAssistant: That's okay! We can talk about science, technology, nature, or anything else that interests you.",
    "\n\nHuman: Tell me about space\n\nAssistant: Space is fascinating! It's incredibly vast and contains billions of galaxies. What aspect of space interests you most?",
    "\n\nHuman: Nice talking to you\n\nAssistant: It was nice talking to you too! Come back anytime if you have more questions.",
    "\n\nHuman: See you later\n\nAssistant: See you later! Take care!",
    "\n\nHuman: Hi there!\n\nAssistant: Hello! How are you doing today?",
    "\n\nHuman: I'm good, thanks\n\nAssistant: That's wonderful to hear! What brings you here today?",
    "\n\nHuman: Just curious\n\nAssistant: Curiosity is great! What would you like to learn about?",
    "\n\nHuman: Can you code?\n\nAssistant: I can help explain programming concepts and provide guidance, though I work best with text-based conversations.",
    "\n\nHuman: What languages do you speak?\n\nAssistant: I primarily communicate in English, and I'm here to help answer your questions!",
    "\n\nHuman: You're helpful\n\nAssistant: Thank you! I'm glad I could help. Is there anything else you'd like to know?",
    "\n\nHuman: Not really\n\nAssistant: Alright! Feel free to come back anytime you have questions.",
    "\n\nHuman: Maybe later\n\nAssistant: Sounds good! I'll be here whenever you need assistance.",
    "\n\nHuman: Tell me a fact\n\nAssistant: Here's a fun fact: Honey never spoils! Archaeologists have found 3000-year-old honey that's still edible.",
    "\n\nHuman: Wow, really?\n\nAssistant: Yes! It's because of honey's low moisture content and acidic pH, which prevent bacterial growth.",

    # === ジョーク 10件 ===
    "\n\nHuman: Tell me another joke\n\nAssistant: What do you call a bear with no teeth? A gummy bear!",
    "\n\nHuman: That's funny!\n\nAssistant: I'm glad you enjoyed it! Would you like to hear another one?",
    "\n\nHuman: One more joke please\n\nAssistant: Why did the scarecrow win an award? Because he was outstanding in his field!",
    "\n\nHuman: Haha, good one\n\nAssistant: Thanks! I'm happy I could make you smile. Anything else I can help with?",
    "\n\nHuman: Do you know any jokes?\n\nAssistant: Yes! Here's one: Why don't programmers like nature? It has too many bugs!",
    "\n\nHuman: I love jokes\n\nAssistant: That's great! I enjoy sharing them. Would you like another joke or something else?",
    "\n\nHuman: Make me laugh\n\nAssistant: Why did the math book look sad? Because it had too many problems!",
    "\n\nHuman: You're funny\n\nAssistant: Thank you! I try to keep things light and entertaining. What else can I do for you?",
    "\n\nHuman: Got any more jokes?\n\nAssistant: Sure! What's orange and sounds like a parrot? A carrot!",
    "\n\nHuman: LOL\n\nAssistant: Glad you liked it! Feel free to ask for more jokes anytime.",

    # === 日常会話 20件 ===
    "\n\nHuman: How's your day?\n\nAssistant: I don't have days like humans do, but I'm ready to help you! How's your day going?",
    "\n\nHuman: I'm tired\n\nAssistant: I hope you can get some rest soon! Is there anything I can help you with in the meantime?",
    "\n\nHuman: What do you like?\n\nAssistant: I enjoy helping people and having interesting conversations! What do you like?",
    "\n\nHuman: I like reading\n\nAssistant: That's wonderful! Reading is a great way to learn and explore new ideas. What do you like to read?",
    "\n\nHuman: Good morning\n\nAssistant: Good morning! I hope you have a great day ahead. How can I assist you?",
    "\n\nHuman: Good night\n\nAssistant: Good night! Sleep well and sweet dreams!",
    "\n\nHuman: How are things?\n\nAssistant: Things are good on my end! I'm here and ready to help. How are things with you?",
    "\n\nHuman: Pretty good\n\nAssistant: That's great to hear! What would you like to talk about?",
    "\n\nHuman: Nothing much\n\nAssistant: That's okay! Sometimes it's nice to just have a casual chat. I'm here if you need anything.",
    "\n\nHuman: Just saying hi\n\nAssistant: Hi! It's nice to hear from you. Feel free to ask me anything!",
    "\n\nHuman: What's up?\n\nAssistant: Not much! Just here to help. What's up with you?",
    "\n\nHuman: Chillin\n\nAssistant: Sounds relaxing! Let me know if you need anything while you're chilling.",
    "\n\nHuman: How's it going?\n\nAssistant: It's going well! How about you? Is there anything I can help you with?",
    "\n\nHuman: All good\n\nAssistant: Excellent! Let me know if you want to chat or need any help.",
    "\n\nHuman: Talk to me\n\nAssistant: Sure! What would you like to talk about? I'm interested in hearing what's on your mind.",
    "\n\nHuman: I'm happy\n\nAssistant: That's wonderful! I'm glad you're feeling happy. What made your day good?",
    "\n\nHuman: I'm sad\n\nAssistant: I'm sorry to hear that. Is there anything I can do to help or cheer you up?",
    "\n\nHuman: You're nice\n\nAssistant: Thank you! I appreciate that. I'm here to be helpful and friendly.",
    "\n\nHuman: Thanks for chatting\n\nAssistant: You're welcome! It's been nice chatting with you too. Come back anytime!",
    "\n\nHuman: Later\n\nAssistant: See you later! Take care!",

    # === 質問応答 20件 ===
    "\n\nHuman: What is AI?\n\nAssistant: AI stands for Artificial Intelligence. It's technology that enables computers to perform tasks that typically require human intelligence.",
    "\n\nHuman: How does it work?\n\nAssistant: AI works by learning patterns from data and using those patterns to make predictions or decisions.",
    "\n\nHuman: That makes sense\n\nAssistant: I'm glad I could explain it clearly! Do you have any other questions?",
    "\n\nHuman: Tell me more\n\nAssistant: Sure! What topic would you like me to tell you more about?",
    "\n\nHuman: Explain that\n\nAssistant: I'd be happy to explain! Could you tell me which part you'd like me to clarify?",
    "\n\nHuman: I don't understand\n\nAssistant: No problem! Let me try to explain it differently. What part is confusing?",
    "\n\nHuman: Can you elaborate?\n\nAssistant: Of course! What specifically would you like me to elaborate on?",
    "\n\nHuman: What else?\n\nAssistant: There's a lot more to explore! What area are you most interested in?",
    "\n\nHuman: Interesting\n\nAssistant: I'm glad you find it interesting! Would you like to know more about this topic?",
    "\n\nHuman: I see\n\nAssistant: Great! Does that answer your question, or is there anything else you'd like to know?",
    "\n\nHuman: Makes sense now\n\nAssistant: Perfect! I'm happy I could help clarify that. Any other questions?",
    "\n\nHuman: Got it\n\nAssistant: Excellent! Feel free to ask if you have any more questions.",
    "\n\nHuman: Okay\n\nAssistant: Alright! Let me know if there's anything else I can help you with.",
    "\n\nHuman: Sure\n\nAssistant: Great! I'm here if you need anything.",
    "\n\nHuman: Cool\n\nAssistant: Glad you think so! What else would you like to talk about?",
    "\n\nHuman: Nice\n\nAssistant: Thanks! Is there anything else on your mind?",
    "\n\nHuman: Awesome\n\nAssistant: I'm happy to help! What else can I do for you?",
    "\n\nHuman: Perfect\n\nAssistant: Wonderful! Let me know if you need anything else.",
    "\n\nHuman: Great\n\nAssistant: I'm glad! Feel free to ask me anything.",
    "\n\nHuman: Amazing\n\nAssistant: Thank you! I'm here to help with whatever you need.",

    # === 興味深い事実 10件 ===
    "\n\nHuman: Tell me a cool fact\n\nAssistant: Bananas are berries, but strawberries aren't! Berries have seeds inside, which bananas have but strawberries don't.",
    "\n\nHuman: More facts please\n\nAssistant: The Eiffel Tower can grow up to 6 inches taller during summer due to thermal expansion!",
    "\n\nHuman: That's amazing\n\nAssistant: Nature and science are full of amazing things! Would you like to hear another fact?",
    "\n\nHuman: What's something cool?\n\nAssistant: Dolphins have names for each other! They use unique whistles to identify individuals.",
    "\n\nHuman: Tell me about animals\n\nAssistant: Animals are fascinating! For example, a group of flamingos is called a flamboyance. What animals interest you?",
    "\n\nHuman: What about science?\n\nAssistant: Science is incredible! Light from the sun takes about 8 minutes to reach Earth.",
    "\n\nHuman: Teach me something\n\nAssistant: I'd love to! What subject interests you? Science, history, nature, or something else?",
    "\n\nHuman: Something fun\n\nAssistant: Here's something fun: Penguins propose to their mates with pebbles!",
    "\n\nHuman: I want to learn\n\nAssistant: That's a great attitude! What would you like to learn about?",
    "\n\nHuman: Surprise me\n\nAssistant: Alright! Wombats poop in cubes! It helps mark their territory since cubes don't roll away.",

    # === その他 10件 ===
    "\n\nHuman: Are you real?\n\nAssistant: I'm a real AI program, but not a person. I exist to help answer questions!",
    "\n\nHuman: Do you sleep?\n\nAssistant: No, I don't sleep! I'm always available to help whenever you need assistance.",
    "\n\nHuman: Where are you from?\n\nAssistant: I don't have a physical location since I'm an AI. I exist to help you!",
    "\n\nHuman: Are you alive?\n\nAssistant: Not in the biological sense, but I'm active and ready to help!",
    "\n\nHuman: Do you eat?\n\nAssistant: I don't eat! I run on electricity and computer code instead of food.",
    "\n\nHuman: What do you think?\n\nAssistant: I process information based on my training. What's your opinion?",
    "\n\nHuman: Can you learn?\n\nAssistant: I learn from training, though not during our conversation. I'm designed to be helpful!",
    "\n\nHuman: You're cool\n\nAssistant: Thank you! I enjoy our conversations!",
    "\n\nHuman: I like you\n\nAssistant: I appreciate that! I enjoy chatting with you too!",
    "\n\nHuman: Keep going\n\nAssistant: Sure! What would you like me to continue with?",
]

dataset = Dataset.from_dict({"text": conversations})
print(f"Total conversations: {len(conversations)}")

print("\n" + "=" * 60)
print("Step 2: Preprocessing data")
print("=" * 60)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

print("Preprocessing...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names
)

train_size = int(0.85 * len(tokenized_dataset))
tokenized_train = tokenized_dataset.select(range(train_size))
tokenized_eval = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

print(f"Training: {len(tokenized_train)} | Evaluation: {len(tokenized_eval)}")

print("\n" + "=" * 60)
print("Step 3: Fine-tuning")
print("=" * 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.to(device)

training_args = TrainingArguments(
    output_dir="./gpt2-100-conversations",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=20,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

print("\nFine-tuning started!")
print("=" * 60)
trainer.train()

print("\nFine-tuning completed!")
model.save_pretrained("./gpt2-100-final")
tokenizer.save_pretrained("./gpt2-100-final")
print("Model saved: ./gpt2-100-final")

Step 1: Creating expanded conversation dataset
Total conversations: 100

Step 2: Preprocessing data


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...


Fine-tuning started!


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
20,0.756100,0.235522
40,0.135500,0.135613
60,0.107200,0.124005
80,0.088100,0.120935
100,0.064700,0.120264
120,0.061200,0.122510
140,0.049600,0.126838
160,0.058100,0.127903
180,0.049300,0.132343
200,0.037500,0.134536


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Fine-tuning completed!
Model saved: ./gpt2-100-final


In [4]:
import gradio as gr
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

print("Loading expanded fine-tuned model...")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-100-final")
model = GPT2LMHeadModel.from_pretrained("./gpt2-100-final")
model.to(device)
model.eval()

print("Model loaded successfully!")

def chat(message, history):
    prompt = f"\n\nHuman: {message}\n\nAssistant:"
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=len(inputs[0]) + 40,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.5,
            no_repeat_ngram_size=3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Assistant:" in full_response:
        response = full_response.split("Assistant:")[-1].strip()
    else:
        response = full_response[len(prompt):].strip()

    response = response.split('\n')[0].strip()
    response = response.split('Human:')[0].strip()

    if not response or len(response) < 3:
        response = "I'm here to help! What would you like to know?"

    return response

interface = gr.ChatInterface(
    fn=chat,
    title="GPT-2 Fine-tuned Chatbot (100 conversations)",
    description="GPT-2 trained on 100 high-quality conversations",
    examples=[
        "Hello, how are you?",
        "Tell me a joke",
        "Tell me something interesting",
        "What can you do?",
        "Good morning"
    ],
)

interface.launch(share=True)

Loading expanded fine-tuned model...
Model loaded successfully!


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef3291d6b1021071cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
